In [1]:
# Import libraries and configure autoreload
from pandas_openscm.io import load_timeseries_csv
import pandas as pd

from gcages import renaming
from gcages.cmip7_scenariomip.harmonisation import (
    create_cmip7_scenariomip_global_harmoniser
)

from pathlib import Path

%load_ext autoreload
%autoreload 2

Read in the scenario data that we have prepared in the notebook `100_prepare_sci_data.ipynb`

In [2]:
# Load prepared scenario data from previous notebook
start = load_timeseries_csv(
    Path("output/100_inputdata_20260428.csv"),
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [3]:
# Define paths to historical emissions and harmonization override files
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS_FILE = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/history_cmip7_scenariomip.csv"
)
ANERIS_GLOBAL_OVERRIDES_FILE = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/aneris-overrides-global.csv"
)

In [4]:
# Create global harmonizer (initial attempt, run serially)
harmoniser_global = create_cmip7_scenariomip_global_harmoniser(
    cmip7_scenariomip_global_historical_emissions_file=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS_FILE,
    aneris_global_overrides_file=ANERIS_GLOBAL_OVERRIDES_FILE,
    n_processes=None,  # run serially for this demo,
)

In [5]:
# Inspect available variables in the input data
start.index.get_level_values('variable').unique()

Index(['Emissions|BC', 'Emissions|CH4', 'Emissions|CO', 'Emissions|CO2|AFOLU',
       'Emissions|CO2|Energy and Industrial Processes', 'Emissions|HFC|HFC125',
       'Emissions|HFC|HFC134a', 'Emissions|HFC|HFC143a',
       'Emissions|HFC|HFC227ea', 'Emissions|HFC|HFC23', 'Emissions|HFC|HFC32',
       'Emissions|HFC|HFC43-10', 'Emissions|N2O', 'Emissions|NH3',
       'Emissions|NOx', 'Emissions|OC', 'Emissions|SF6', 'Emissions|Sulfur',
       'Emissions|VOC', 'Emissions|C2F6', 'Emissions|C6F14', 'Emissions|CF4',
       'Emissions|HFC|HFC245fa'],
      dtype='object', name='variable')

In [6]:
# Rename variables from CMIP7 ScenarioMIP to gcages convention
start_renamed = renaming.rename_variables(
    start,
    from_convention=renaming.SupportedNamingConventions.CMIP7_SCENARIOMIP,
    to_convention=renaming.SupportedNamingConventions.GCAGES
)

In [7]:
# Fix HFC43-10 unit naming issue (remove hyphen)
start_renamed = start_renamed.reset_index()
start_renamed['unit'] = start_renamed['unit'].apply(
    lambda x: 'kt HFC4310/yr' if x=='kt HFC43-10/yr' else x
)
start_renamed.set_index(["model", "scenario", "region", "variable", "unit"], inplace=True)

In [8]:
# Create harmonizer without checks
harmoniser_global = create_cmip7_scenariomip_global_harmoniser(
    CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS_FILE,
    ANERIS_GLOBAL_OVERRIDES_FILE,
    run_checks=False
)

In [9]:
# Harmonize the scenario data to historical emissions
harmonised_global = harmoniser_global(start_renamed)

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
1522it [00:00, 9101.94it/s]
Scenarios to harmonise:   0%|          | 0/1522 [00:00<?, ?it/s]INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with constant_ratio
Scenarios to harmonise:   0%|          | 1/1522 [00:00<10:59,  2.31it/s]INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with constant_rat

In [10]:
# View harmonized results
harmonised_global

year                                                                                     2023  \
model             scenario             region variable                unit                      
AIM/CGE 2.0       SSP1-26              World  Emissions|BC            Mt BC/yr       7.219245   
                                              Emissions|CH4           Mt CH4/yr    363.489901   
                                              Emissions|CO            Mt CO/yr     774.053434   
                                              Emissions|CO2|Biosphere Mt CO2/yr   3627.550667   
                                              Emissions|CO2|Fossil    Mt CO2/yr  38712.217040   
...                                                                                       ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi2020_400 World  Emissions|CH4           Mt CH4/yr    363.489901   
                                              Emissions|CO2|Biosphere Mt CO2/yr   3627.550667   
                                              Emissions|CO2|Fossil    Mt CO2/yr  38712.217040   
                                              Emissions|N2O           kt N2O/yr  11289.003671   
                                              Emissions|SOx           Mt SO2/yr     71.902038   

year                                                                                     2024  \
model             scenario             region variable                unit                      
AIM/CGE 2.0       SSP1-26              World  Emissions|BC            Mt BC/yr       6.980383   
                                              Emissions|CH4           Mt CH4/yr    346.465510   
                                              Emissions|CO            Mt CO/yr     758.753924   
                                              Emissions|CO2|Biosphere Mt CO2/yr   3317.876430   
                                              Emissions|CO2|Fossil    Mt CO2/yr  38076.503815   
...                                                                                       ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi2020_400 World  Emissions|CH4           Mt CH4/yr    302.563268   
                                              Emissions|CO2|Biosphere Mt CO2/yr   3003.588476   
                                              Emissions|CO2|Fossil    Mt CO2/yr  35440.590135   
                                              Emissions|N2O           kt N2O/yr  10376.319224   
                                              Emissions|SOx           Mt SO2/yr     65.928851   

year                                                                                     2025  \
model             scenario             region variable                unit                      
AIM/CGE 2.0       SSP1-26              World  Emissions|BC            Mt BC/yr       6.743266   
                                              Emissions|CH4           Mt CH4/yr    329.489904   
                                              Emissions|CO            Mt CO/yr     743.486145   
                                              Emissions|CO2|Biosphere Mt CO2/yr   3008.202193   
                                              Emissions|CO2|Fossil    Mt CO2/yr  37443.855749   
...                                                                                       ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi2020_400 World  Emissions|CH4           Mt CH4/yr    242.235936   
                                              Emissions|CO2|Biosphere Mt CO2/yr   2379.626285   
                                              Emissions|CO2|Fossil    Mt CO2/yr  32192.171307   
                                              Emissions|N2O           kt N2O/yr   9470.930453   
                                              Emissions|SOx           Mt SO2/yr     59.974828   

year                                                                                     2026  \
model             scenario             region variable                unit                      
AIM/CGE 2.0       SS

In [11]:
# Rename variables back to CMIP7 ScenarioMIP convention
final_renaming = renaming.rename_variables(
    harmonised_global,
    to_convention=renaming.SupportedNamingConventions.CMIP7_SCENARIOMIP,
    from_convention=renaming.SupportedNamingConventions.GCAGES
)

In [12]:
# View final renamed results
final_renaming.to_csv(
    Path('output/101_harmonised_data.csv')
)